In [ ]:
!pip install pysam biopython pymemesuite

In [ ]:
!apt-get update
!apt-get install -y samtools

In [ ]:
!wget https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz
!gunzip hg38.fa.gz

In [ ]:
!samtools faidx hg38.fa

In [ ]:
!wget https://jaspar.elixir.no/download/data/2024/CORE/JASPAR2024_CORE_vertebrates_non-redundant_pfms_meme.txt -O motifs.meme

In [ ]:
import pandas as pd
import pysam
from scipy.stats import fisher_exact
from pymemesuite.fimo import FIMO
from pymemesuite.common import MotifFile, Sequence

In [ ]:
# === CONFIGURATION ===
VCF_PATH = "/content/high_rare_regulatory.vcf.gz"
FASTA_PATH = "/content/hg38.fa"
MOTIF_PATH = "/content/motifs.meme"
FLANK_SIZE = 30
BATCH_SIZE = 5000

N_PRIORITIZED = 16

# strict uppercase keys to protect mapping
PRIORITIZED_HITS = {
    "EWSR1-FLI1": 3,
    "FOXH1": 1,
    "INSM1": 1,
    "IRF2": 1,
    "Lhx3": 1,
    "NFIA": 1,
    "NFIC::TLX1": 1,
    "NR1I2": 1,
    "PAX9": 1,
    "RARA": 1,
    "RELA": 1,
    "Stat5a::Stat5b": 1,
    "ZNF354C": 2,
}

# === 1. LOAD MOTIFS — Precise Key Resolution ===
motif_dict = {}
with MotifFile(MOTIF_PATH) as mf:
    for m in mf:
        names_to_try = []
        if m.name:
            names_to_try.append(m.name.decode().strip().upper())

        for attr in ["name2", "label", "alt_name"]:
            val = getattr(m, attr, None)
            if val:
                names_to_try.append(val.decode().strip().upper())
                break

        # Cross-reference with our target prioritized TFs
        for target_tf in PRIORITIZED_HITS.keys():
            t_up = target_tf.upper()
            # Generate expected mutations for composite names
            variants = [t_up, t_up.replace('::', '-'), t_up.replace('::', '_'), t_up.replace('-', '::')]

            if any(v in names_to_try for v in variants):
                motif_dict[target_tf] = m

    bg_model = mf.background

# Sanity Check
print("TF lookup results:")
for tf in PRIORITIZED_HITS:
    found = tf in motif_dict
    print(f"  {tf}: {'FOUND' if found else 'NOT FOUND — skipping enrichment'}")

# === 2. BATCH SCANNING FUNCTION ===
fimo = FIMO(both_strands=True, threshold=1e-3)
bg_counts = {tf: 0 for tf in PRIORITIZED_HITS.keys()}
total_bg_count = 0

def process_batch(batch_sequences):
    """Scores a sequence batch against all successfully found motifs."""
    if not batch_sequences:
        return
    for tf_name, motif in motif_dict.items():
        res = fimo.score_motif(motif, batch_sequences, bg_model)
        # Unique ID assignment tracking via sequence headers
        hit_ids = {m.source.accession.decode() for m in res.matched_elements}
        bg_counts[tf_name] += len(hit_ids)

# === 3. STREAM VCF & SCORE ===
print("\nScanning background variants...")
fasta = pysam.FastaFile(FASTA_PATH)
vcf = pysam.VariantFile(VCF_PATH)

current_batch = []

for record in vcf:
    if "PASS" not in record.filter:
        continue

    chrom = record.chrom
    pos = record.pos
    ref = record.ref

    # Safely extract alternative allele string
    if not record.alts:
        continue
    alt = record.alts[0]

    try:
        # Check boundary sizes using fasta reference safety lengths
        chrom_len = fasta.get_reference_length(chrom)
        start_idx = pos - 1 - FLANK_SIZE
        end_idx = pos - 1 + len(ref) + FLANK_SIZE

        if start_idx < 0 or end_idx > chrom_len:
            continue

        left = fasta.fetch(chrom, start_idx, pos - 1)
        right = fasta.fetch(chrom, pos - 1 + len(ref), end_idx)
        seq_str = left + alt + right

        # Local batch scoping tracking index (0 to BATCH_SIZE - 1)
        idx_str = f"b_{len(current_batch)}".encode()
        current_batch.append(Sequence(seq_str, name=idx_str))
        total_bg_count += 1

    except Exception:
        continue

    if len(current_batch) >= BATCH_SIZE:
        process_batch(current_batch)
        current_batch = []
        print(f"Processed {total_bg_count} genomic variants...")

# Flush leftover data structures
if current_batch:
    process_batch(current_batch)
print(f"Completed pipeline run. Grand total variants evaluated: {total_bg_count}")

# === 4. STATISTICAL CALCULATIONS ===
results = []
for tf, test_hits in PRIORITIZED_HITS.items():
    if tf not in motif_dict:
        continue

    bg_hits = bg_counts[tf]

    # Contingency matrix setup
    table = [
        [test_hits, N_PRIORITIZED - test_hits],
        [bg_hits, max(0, total_bg_count - bg_hits)]
    ]

    odds, pval = fisher_exact(table, alternative='greater')

    results.append({
        "TF": tf,
        "Test_Rate": f"{test_hits}/{N_PRIORITIZED}",
        "BG_Rate": f"{bg_hits}/{total_bg_count}",
        "BG_Perc": f"{(bg_hits / max(1, total_bg_count)) * 100:.2f}%",
        "Odds_Ratio": round(odds, 3) if pd.notnull(odds) else 0.0,
        "P_Value": pval
    })

df_final = pd.DataFrame(results).sort_values("Odds_Ratio", ascending=False)
print("\n--- FINAL GENOME-WIDE ENRICHMENT ---")
print(df_final.to_string(index=False))

In [ ]:
from statsmodels.stats.multitest import multipletests

df_final["FDR"] = multipletests(
    df_final["P_Value"],
    method="fdr_bh"
)[1]

df_final = df_final.sort_values("FDR")

print(df_final.to_string(index=False))

low-infectivity variants

In [ ]:
# === CONFIGURATION ===
VCF_PATH = "/content/low_rare_regulatory.vcf.gz"
FASTA_PATH = "/content/hg38.fa"
MOTIF_PATH = "/content/motifs.meme"
FLANK_SIZE = 30
BATCH_SIZE = 5000

N_PRIORITIZED = 10

PRIORITIZED_HITS = {
    "BCL6B": 2,
    "Gfi1B": 1,
    "HES1": 1,
    "Lhx3": 1,
    "LMX1B": 1,
    "PLAGL2": 1,
    "Smad4": 1,
    "TLX2": 1,
    "Znf423": 1,
}


# === 1. LOAD MOTIFS — Precise Key Resolution ===
motif_dict = {}
with MotifFile(MOTIF_PATH) as mf:
    for m in mf:
        names_to_try = []
        if m.name:
            names_to_try.append(m.name.decode().strip().upper())

        for attr in ["name2", "label", "alt_name"]:
            val = getattr(m, attr, None)
            if val:
                names_to_try.append(val.decode().strip().upper())
                break

        # Cross-reference with our target prioritized TFs
        for target_tf in PRIORITIZED_HITS.keys():
            t_up = target_tf.upper()
            # Generate expected mutations for composite names
            variants = [t_up, t_up.replace('::', '-'), t_up.replace('::', '_'), t_up.replace('-', '::')]

            if any(v in names_to_try for v in variants):
                motif_dict[target_tf] = m

    bg_model = mf.background

# Sanity Check
print("TF lookup results:")
for tf in PRIORITIZED_HITS:
    found = tf in motif_dict
    print(f"  {tf}: {'FOUND' if found else 'NOT FOUND — skipping enrichment'}")

# === 2. BATCH SCANNING FUNCTION ===
fimo = FIMO(both_strands=True, threshold=1e-3)
bg_counts = {tf: 0 for tf in PRIORITIZED_HITS.keys()}
total_bg_count = 0

def process_batch(batch_sequences):
    """Scores a sequence batch against all successfully found motifs."""
    if not batch_sequences:
        return
    for tf_name, motif in motif_dict.items():
        res = fimo.score_motif(motif, batch_sequences, bg_model)
        # Unique ID assignment tracking via sequence headers
        hit_ids = {m.source.accession.decode() for m in res.matched_elements}
        bg_counts[tf_name] += len(hit_ids)

# === 3. STREAM VCF & SCORE ===
print("\nScanning background variants...")
fasta = pysam.FastaFile(FASTA_PATH)
vcf = pysam.VariantFile(VCF_PATH)

current_batch = []

for record in vcf:
    if "PASS" not in record.filter:
        continue

    chrom = record.chrom
    pos = record.pos
    ref = record.ref

    # Safely extract alternative allele string
    if not record.alts:
        continue
    alt = record.alts[0]

    try:
        # Check boundary sizes using fasta reference safety lengths
        chrom_len = fasta.get_reference_length(chrom)
        start_idx = pos - 1 - FLANK_SIZE
        end_idx = pos - 1 + len(ref) + FLANK_SIZE

        if start_idx < 0 or end_idx > chrom_len:
            continue

        left = fasta.fetch(chrom, start_idx, pos - 1)
        right = fasta.fetch(chrom, pos - 1 + len(ref), end_idx)
        seq_str = left + alt + right

        # Local batch scoping tracking index (0 to BATCH_SIZE - 1)
        idx_str = f"b_{len(current_batch)}".encode()
        current_batch.append(Sequence(seq_str, name=idx_str))
        total_bg_count += 1

    except Exception:
        continue

    if len(current_batch) >= BATCH_SIZE:
        process_batch(current_batch)
        current_batch = []
        print(f"Processed {total_bg_count} genomic variants...")

# Flush leftover data structures
if current_batch:
    process_batch(current_batch)
print(f"Completed pipeline run. Grand total variants evaluated: {total_bg_count}")

# === 4. STATISTICAL CALCULATIONS ===
results = []
for tf, test_hits in PRIORITIZED_HITS.items():
    if tf not in motif_dict:
        continue

    bg_hits = bg_counts[tf]

    # Contingency matrix setup
    table = [
        [test_hits, N_PRIORITIZED - test_hits],
        [bg_hits, max(0, total_bg_count - bg_hits)]
    ]

    odds, pval = fisher_exact(table, alternative='greater')

    results.append({
        "TF": tf,
        "Test_Rate": f"{test_hits}/{N_PRIORITIZED}",
        "BG_Rate": f"{bg_hits}/{total_bg_count}",
        "BG_Perc": f"{(bg_hits / max(1, total_bg_count)) * 100:.2f}%",
        "Odds_Ratio": round(odds, 3) if pd.notnull(odds) else 0.0,
        "P_Value": pval
    })

df_final = pd.DataFrame(results).sort_values("Odds_Ratio", ascending=False)
print("\n--- FINAL GENOME-WIDE ENRICHMENT ---")
print(df_final.to_string(index=False))

In [ ]:
from statsmodels.stats.multitest import multipletests

df_final["FDR"] = multipletests(
    df_final["P_Value"],
    method="fdr_bh"
)[1]

df_final = df_final.sort_values("FDR")

print(df_final.to_string(index=False))